# 🎵 AudioLab / VocalCut Studio - Free GPU AI Backend
Run Meta HTDemucs AI on a **FREE NVIDIA T4 GPU** (Google Colab) and connect it to your web app!
Takes ~8 seconds to separate an entire song!

In [ ]:
# Step 1: Install Demucs, FastAPI, Uvicorn, and Localtunnel
!pip install -q demucs fastapi uvicorn python-multipart
!npm install -g localtunnel

In [ ]:
# Step 2: Write FastAPI Server code
import os

code = '''
import os, shutil, uuid, base64, subprocess
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import torch

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
TEMP_DIR = Path("./temp_audio")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

@app.get("/")
@app.get("/api/health")
def health():
    return {"status": "online", "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/api/separate")
async def separate(file: UploadFile = File(...)):
    job_id = str(uuid.uuid4())
    job_dir = TEMP_DIR / job_id
    job_dir.mkdir(parents=True, exist_ok=True)
    input_path = job_dir / "input.mp3"
    try:
        with open(input_path, "wb") as f:
            shutil.copyfileobj(file.file, f)
        out_dir = job_dir / "out"
        cmd = ["python", "-m", "demucs.separate", "-n", "htdemucs", "--two-stems=vocals", "-d", "cuda" if torch.cuda.is_available() else "cpu", "-o", str(out_dir), str(input_path)]
        subprocess.run(cmd, check=True)
        res_dir = out_dir / "htdemucs" / "input"
        with open(res_dir / "vocals.wav", "rb") as f:
            voc = base64.b64encode(f.read()).decode()
        with open(res_dir / "no_vocals.wav", "rb") as f:
            inst = base64.b64encode(f.read()).decode()
        return JSONResponse({"status": "success", "vocals_base64": f"data:audio/wav;base64,{voc}", "instrumental_base64": f"data:audio/wav;base64,{inst}"})
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)
'''

with open('colab_server.py', 'w') as f:
    f.write(code)
print('Backend server written successfully!')

In [ ]:
# Step 3: Start Server & Generate Public HTTPS URL
import subprocess, time, urllib.request

# Start uvicorn in background
subprocess.Popen(["uvicorn", "colab_server:app", "--port", "7860"])
time.sleep(3)

# Get external IP for localtunnel password
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"\n👉 Tunnel Password (if prompted): {ip}\n")

# Start Localtunnel and print public URL
!npx localtunnel --port 7860